# Experiment 01 — RAFT-Sintel master baseline

**Flow State · RoCo-45 · Optical Flow**

This Kaggle notebook performs **inference only** on the clean Spring test split and creates an upload-ready benchmark submission. Its fixed baseline is **full RAFT**, the official **Sintel-pretrained checkpoint**, **32 recurrent refinement iterations**, **native 1920×1080 input**, **Triton correlation**, and **two T4 GPUs**. It validates the prediction tree and runs the official `flow_subsampling` executable.

The Spring test split has no public optical-flow ground truth, so this notebook cannot calculate EPE or WAUC locally. Accuracy comes from the benchmark after upload. The local checks cover configuration identity, image layout, output count, filenames, tensor shapes, numerical validity, and packaging.

The notebook intentionally does not run RobustSpring's 20 corruptions; that workload is not safe to combine with native-resolution clean inference in one 12-hour Kaggle session.

## Model used: full RAFT with Sintel weights

**RAFT (Recurrent All-Pairs Field Transforms)** extracts a feature map from each frame, constructs a multiscale correlation representation between the two feature maps, and recurrently updates a dense 2-D flow field. This devkit configuration uses the full RAFT network—not `raft_small`—with a 256-channel feature encoder, four correlation-pyramid levels, correlation radius four, a recurrent update block, and learned convex upsampling.

The checkpoint is `raft-sintel-fb44381e.ckpt` from the official PTLFlow release. It has been pretrained/fine-tuned for MPI Sintel and is used without Spring fine-tuning, making this a reproducible transfer baseline. `--model.iters 32` applies 32 recurrent flow refinements.

`corr_mode=triton` changes how correlation lookup is computed to make high-resolution inference practical; it does not select a different trained model. The model runs once as frame 1→2 and again with the frames swapped for frame 2→1, producing the required FW and BW outputs for both left and right cameras.

Native resolution means the images are not rescaled. RAFT may temporarily pad an image to a multiple of its stride (8), then removes that padding so every saved flow has shape `1080×1920×2`. Each pixel stores horizontal displacement `u` and vertical displacement `v`.

## What happens from start to finish

1. Define and print the immutable experiment configuration.
2. Confirm that Kaggle provided exactly two T4 GPUs and inspect available disk.
3. Locate the extracted Spring test data and verify both camera streams.
4. Install a pinned official devkit revision and verify the RAFT class defaults.
5. Locate the Sintel weights and official subsampling executable.
6. Replicate RAFT onto the two GPUs and shard independent frame pairs between them.
7. Write native-resolution `.flo5` predictions into temporary storage.
8. Verify every expected output before running the official packager.
9. Save the submission HDF5 and a reproducibility manifest in `/kaggle/working`.

## Required Kaggle inputs and official links

Prefer extracting the large archives **before** uploading them as Kaggle Datasets, then attach them through **Add Input**. The notebook also accepts separately attached left/right folders or the original ZIP files; ZIP inputs are extracted into `/kaggle/temp` and therefore cost session time. `/kaggle/input` is read-only, predictions go to disposable `/kaggle/temp`, and only the packaged artifact is kept in `/kaggle/working`.

1. [Spring dataset — DaRUS DOI 10.18419/DARUS-3376](https://darus.uni-stuttgart.de/dataset.xhtml?persistentId=doi:10.18419/DARUS-3376)
   - `test_frame_left.zip`
   - `test_frame_right.zip`
   - `test_cam_data.zip`
2. [Official RoCo-Spring devkit](https://github.com/hmorimitsu/roco-spring-devkit) — optional as an attached input when Kaggle Internet is off.
3. [Official RAFT-Sintel checkpoint](https://github.com/hmorimitsu/ptlflow/releases/download/weights1/raft-sintel-fb44381e.ckpt) — attach this file when Internet is off.
4. [Official Spring subsampling tools](https://cloud.visus.uni-stuttgart.de/index.php/s/J33vJ3meZf7Jeq1) — attach the extracted Linux `flow_subsampling` executable.
5. [Challenge participation instructions](https://roco-spring.github.io/participate.html) and [Spring benchmark submission site](https://spring-benchmark.org/).

In **Settings → Accelerator**, select **GPU T4 ×2**. Keeping Kaggle Internet enabled is recommended so `pip` can resolve any missing packages. Internet-off execution requires the devkit, checkpoint, and all Python dependencies to already be available as attached inputs or in the runtime image.

## 1. Freeze the experiment configuration

This cell records the team identity and all variables that define Experiment 01. Large raw predictions are sent to `/kaggle/temp` because they are disposable and can exceed Kaggle's saved-output allowance. Only the compact packaged submission and manifest go to `/kaggle/working`. Do not change `ITERATIONS`, `CHECKPOINT_ALIAS`, or resolution controls if you want results comparable to this master baseline.

In [1]:
from pathlib import Path
import hashlib, importlib, json, os, platform, re, shutil, subprocess, sys, time

TEAM_ID = "RoCo-45"
TEAM_NAME = "Flow State"
MODEL = "raft"
CHECKPOINT_ALIAS = "sintel"
ITERATIONS = 32
CORR_MODE = "triton"
NUM_GPUS = 2
MAX_FORWARD_SIDE = None  # None is mandatory for this native-resolution experiment.
SAVE_VISUALIZATIONS = False
DEVKIT_REF = "90ae81a9324c6806dc3c2482aab84a2744215bd9"

KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
SCRATCH_ROOT = Path("/kaggle/temp")
DEVKIT_DIR = WORK_ROOT / "roco-spring-devkit"
OUTPUT_BASE = SCRATCH_ROOT / "exp01_predictions"
ARTIFACT_DIR = WORK_ROOT / "exp01_artifacts"

# Preferred paths for the Flow State Kaggle datasets. Automatic discovery remains as fallback.
FLOW_STATE_INPUT_ROOT = KAGGLE_INPUT / "datasets/syedmohaiminulhoque"
PREFERRED_SPRING_PARTS = [
    FLOW_STATE_INPUT_ROOT / "test-frame-left-right/test_frame_left/spring",
    FLOW_STATE_INPUT_ROOT / "test-frame-left-right/test_frame_right/spring",
    FLOW_STATE_INPUT_ROOT / "test-frame-left-right/test_cam_data/spring",
]
PREFERRED_CHECKPOINT = FLOW_STATE_INPUT_ROOT / "ckpoint/raft-sintel-fb44381e.ckpt"
PREFERRED_SUBSAMPLING_TOOL = FLOW_STATE_INPUT_ROOT / "flow-subsampling/flow_subsampling"
SESSION_START = time.monotonic()
MAX_SESSION_HOURS = 12.0
PACKAGING_RESERVE_MINUTES = 45

assert KAGGLE_INPUT.is_dir(), "Run this notebook in Kaggle."
experiment_config = {
    "team": f"{TEAM_ID} — {TEAM_NAME}",
    "model": "full RAFT",
    "checkpoint": CHECKPOINT_ALIAS,
    "iterations": ITERATIONS,
    "correlation": CORR_MODE,
    "input_resolution": "native 1920x1080 (no rescaling)",
    "gpus": NUM_GPUS,
    "fine_tuning": False,
    "test_time_augmentation": False,
}
print(json.dumps(experiment_config, indent=2))

{
  "team": "RoCo-45 \u2014 Flow State",
  "model": "full RAFT",
  "checkpoint": "sintel",
  "iterations": 32,
  "correlation": "triton",
  "input_resolution": "native 1920x1080 (no rescaling)",
  "gpus": 2,
  "fine_tuning": false,
  "test_time_augmentation": false
}


## 2. Hardware and storage preflight

Native-resolution RAFT is memory intensive. This check prevents accidentally starting on CPU, one T4, or the wrong accelerator. The official test runner makes one model replica per T4 and assigns each GPU a contiguous shard of samples; this is inference parallelism, not distributed training.

In [2]:
# Hardware and disk preflight. Fail early instead of losing a long session.
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True, capture_output=True, check=True,
)
print(gpu.stdout)
gpu_lines = [x for x in gpu.stdout.splitlines() if x.strip()]
if len(gpu_lines) != 2 or any("T4" not in x for x in gpu_lines):
    raise RuntimeError("Select Kaggle GPU T4 x2 and restart the session.")

for path in (WORK_ROOT, SCRATCH_ROOT):
    path.mkdir(parents=True, exist_ok=True)
    usage = shutil.disk_usage(path)
    print(f"{path}: {usage.free / 1024**3:.1f} GiB free")

print("Python:", sys.version.split()[0], "OS:", platform.platform())

Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

/kaggle/working: 19.5 GiB free
/kaggle/temp: 1026.8 GiB free
Python: 3.12.13 OS: Linux-6.12.90+-x86_64-with-glibc2.35


## 3. Discover and verify Spring test images

The optical-flow test loader needs consecutive PNG frames from `test/####/frame_left` and `frame_right`. The code accepts a unified extracted dataset, separately attached left/right datasets, or the original ZIP files. It builds one canonical tree in `/kaggle/temp` using directory symlinks whenever possible, verifies all ten sequences and matching camera streams, and computes the exact number of flow files inference must produce. Camera metadata is part of the official download but is not consumed by this two-frame RAFT model.

In [3]:
# Locate unified or separately attached inputs without descending into image folders.
import zipfile

EXPECTED_SEQUENCES = {"0003", "0019", "0028", "0029", "0031", "0034", "0035", "0040", "0042", "0046"}
CORRUPTION_NAMES = {
    "brightness", "contrast", "defocus_blur", "elastic_transform", "fog",
    "frost", "gaussian_blur", "gaussian_noise", "glass_blur",
    "impulse_noise", "jpeg_compression", "motion_blur", "pixelate",
    "rain", "saturate", "shot_noise", "snow", "spatter",
    "speckle_noise", "zoom_blur",
}

def metadata_walk(root: Path, max_depth: int = 8):
    root = root.resolve()
    for current, dirs, files in os.walk(root):
        current = Path(current)
        depth = len(current.relative_to(root).parts)
        yield current, list(dirs), list(files)
        # Frame folders contain thousands of PNGs; their names are checked later.
        dirs[:] = [d for d in dirs if d not in {"frame_left", "frame_right"}]
        if depth >= max_depth:
            dirs[:] = []

def attached_files_named(filename: str):
    matches = []
    for current, _, files in metadata_walk(KAGGLE_INPUT):
        if filename in files:
            matches.append(current / filename)
    return sorted(set(matches), key=str)

EXTRACT_ROOT = SCRATCH_ROOT / "exp01_extracted_inputs"
preferred_frames_ready = all(path.is_dir() for path in PREFERRED_SPRING_PARTS[:2])
search_roots = [path for path in PREFERRED_SPRING_PARTS if path.is_dir()] if preferred_frames_ready else [KAGGLE_INPUT]
required_archives = ["test_frame_left.zip", "test_frame_right.zip"]
archive_matches = {name: [] for name in required_archives + ["test_cam_data.zip"]}
if not preferred_frames_ready:
    archive_matches = {name: attached_files_named(name) for name in archive_matches}
if all(archive_matches[name] for name in required_archives):
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    for name, matches in archive_matches.items():
        if not matches:
            continue
        destination = EXTRACT_ROOT / Path(name).stem
        marker = destination / ".extraction_complete"
        if not marker.exists():
            destination.mkdir(parents=True, exist_ok=True)
            print(f"Extracting {matches[0].name} to temporary storage...")
            with zipfile.ZipFile(matches[0]) as archive:
                archive.extractall(destination)
            marker.touch()
    search_roots.append(EXTRACT_ROOT)

side_sources = {"left": {}, "right": {}}
camera_sources = {}
for root in search_roots:
    for current, dirs, _ in metadata_walk(root):
        if current.name not in EXPECTED_SEQUENCES or current.parent.name != "test":
            continue
        if any(part in CORRUPTION_NAMES for part in current.parts):
            continue
        for side in ("left", "right"):
            source = current / f"frame_{side}"
            if source.is_dir() and next(source.glob(f"frame_{side}_*.png"), None) is not None:
                side_sources[side].setdefault(current.name, []).append(source.resolve())
        camera = current / "cam_data"
        if camera.is_dir():
            camera_sources.setdefault(current.name, []).append(camera.resolve())

missing = {side: sorted(EXPECTED_SEQUENCES - set(side_sources[side])) for side in ("left", "right")}
if missing["left"] or missing["right"]:
    diagnostic = {
        "missing_sequences": missing,
        "attached_zip_files": {k: [str(p) for p in v] for k, v in archive_matches.items()},
        "top_level_inputs": sorted(p.name for p in KAGGLE_INPUT.iterdir()),
    }
    raise RuntimeError("Could not assemble Spring test frames:\n" + json.dumps(diagnostic, indent=2))

# Always assemble one canonical root. Directory symlinks avoid copying extracted Kaggle data.
SPRING_ROOT = SCRATCH_ROOT / "exp01_spring_merged"
def ensure_directory_link(source: Path, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_symlink():
        if destination.resolve() != source.resolve():
            raise RuntimeError(f"Conflicting existing link: {destination}")
    elif destination.exists():
        raise RuntimeError(f"Expected a link destination, but a file/directory exists: {destination}")
    else:
        destination.symlink_to(source, target_is_directory=True)

for sequence_name in sorted(EXPECTED_SEQUENCES):
    sequence_root = SPRING_ROOT / "test" / sequence_name
    for side in ("left", "right"):
        source = sorted(set(side_sources[side][sequence_name]), key=str)[0]
        ensure_directory_link(source, sequence_root / f"frame_{side}")
    if sequence_name in camera_sources:
        source = sorted(set(camera_sources[sequence_name]), key=str)[0]
        ensure_directory_link(source, sequence_root / "cam_data")

sequences = sorted((SPRING_ROOT / "test").glob("[0-9][0-9][0-9][0-9]"))
image_count = 0
expected_flow_files = 0
for seq in sequences:
    left = sorted((seq / "frame_left").glob("frame_left_*.png"))
    right = sorted((seq / "frame_right").glob("frame_right_*.png"))
    if not left or len(left) != len(right):
        raise RuntimeError(f"Missing or mismatched frames in {seq}: left={len(left)}, right={len(right)}")
    image_count += len(left) + len(right)
    expected_flow_files += 4 * (len(left) - 1)
print(f"Canonical Spring root: {SPRING_ROOT}")
print(f"Verified {len(sequences)} test sequences and {image_count:,} images")
print(f"Expected prediction files: {expected_flow_files:,}")

Canonical Spring root: /kaggle/temp/exp01_spring_merged
Verified 10 test sequences and 2,000 images
Expected prediction files: 3,960


## 4. Install and inspect the official model implementation

For reproducibility, the notebook pins the same devkit commit used to prepare this experiment. With Internet enabled it clones GitHub; otherwise it copies an attached devkit into writable storage. After installation, the code instantiates a lightweight CPU probe of the exact RAFT class to verify its architecture settings, direction handling, iteration count, and official Sintel checkpoint mapping before spending hours on inference.

In [4]:
# Use an attached devkit if available; otherwise clone the official repository.
def find_attached_devkit():
    for path, _, files in metadata_walk(KAGGLE_INPUT):
        if "pyproject.toml" in files and (path / "roco_spring_devkit" / "optical_flow" / "test.py").is_file():
            return path.resolve()
    return None

if not (DEVKIT_DIR / "roco_spring_devkit" / "optical_flow" / "test.py").is_file():
    attached_devkit = find_attached_devkit()
    if attached_devkit is not None:
        print("Copying attached devkit:", attached_devkit)
        shutil.copytree(attached_devkit, DEVKIT_DIR, dirs_exist_ok=True)
    else:
        print("Cloning official devkit; Kaggle Internet must be enabled.")
        subprocess.run(["git", "clone", "https://github.com/hmorimitsu/roco-spring-devkit.git", str(DEVKIT_DIR)], check=True)

if (DEVKIT_DIR / ".git").is_dir():
    subprocess.run(["git", "checkout", DEVKIT_REF], cwd=DEVKIT_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(DEVKIT_DIR)], check=True)
# Editable-install .pth files are only processed when Python starts. Make this live kernel see the clone now.
devkit_path = str(DEVKIT_DIR.resolve())
if devkit_path not in sys.path:
    sys.path.insert(0, devkit_path)
importlib.invalidate_caches()
print("Devkit ready:", DEVKIT_DIR)

from roco_spring_devkit.optical_flow.models.raft.raft import RAFT
model_probe = RAFT(iters=ITERATIONS, corr_mode=CORR_MODE)
expected_checkpoint_url = "https://github.com/hmorimitsu/ptlflow/releases/download/weights1/raft-sintel-fb44381e.ckpt"
assert RAFT.pretrained_checkpoints[CHECKPOINT_ALIAS] == expected_checkpoint_url
assert model_probe.__class__.__name__ == "RAFT"
assert model_probe.iters == 32
assert model_probe.corr_levels == 4 and model_probe.corr_radius == 4
assert model_probe.predict_all_directions is True
parameter_count = sum(p.numel() for p in model_probe.parameters())
print(f"Verified full RAFT: {parameter_count / 1e6:.2f}M parameters")
print("Correlation pyramid: 4 levels; radius: 4; directions: forward + backward")
print("Official checkpoint URL:", expected_checkpoint_url)
del model_probe

Cloning official devkit; Kaggle Internet must be enabled.


Cloning into '/kaggle/working/roco-spring-devkit'...
Note: switching to '90ae81a9324c6806dc3c2482aab84a2744215bd9'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 90ae81a Merge pull request #3 from hmorimitsu/fixes2


Obtaining file:///kaggle/working/roco-spring-devkit
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 

2026-09-10 04:14:22.318 | WARNING  | roco_spring_devkit.scene_flow.models.raft_3d.raft_3d:<module>:12 - lietorch not found. Please install lietorch from https://github.com/princeton-vl/lietorch if you want to use RAFT3D
2026-09-10 04:14:22.346 | WARNING  | roco_spring_devkit.scene_flow.models.raft_3d.blocks.grid:<module>:13 - scikit-sparse not found. Please install scikit-sparse if you want to use RAFT3D.


Verified full RAFT: 5.26M parameters
Correlation pyramid: 4 levels; radius: 4; directions: forward + backward
Official checkpoint URL: https://github.com/hmorimitsu/ptlflow/releases/download/weights1/raft-sintel-fb44381e.ckpt


## 5. Resolve weights and the official packager

If `raft-sintel-fb44381e.ckpt` is attached, the local path is passed directly to the devkit. Otherwise the devkit resolves the `sintel` alias to the official PTLFlow release URL verified above. The Linux `flow_subsampling` binary is copied from read-only Kaggle Input into `/kaggle/working` so it can be marked executable.

In [5]:
# Prefer the known attached checkpoint for Internet-off execution, then fall back to discovery.
checkpoint_files = [PREFERRED_CHECKPOINT] if PREFERRED_CHECKPOINT.is_file() else attached_files_named("raft-sintel-fb44381e.ckpt")
CKPT_ARG = str(checkpoint_files[0].resolve()) if checkpoint_files else CHECKPOINT_ALIAS
if checkpoint_files:
    if checkpoint_files[0].stat().st_size < 1_000_000:
        raise RuntimeError(f"Checkpoint appears incomplete: {checkpoint_files[0]}")
    print("Attached checkpoint:", CKPT_ARG)
else:
    print("No attached checkpoint found; the official 'sintel' checkpoint will download.")

# Copy the official executable out of read-only /kaggle/input before chmod.
tool_candidates = [PREFERRED_SUBSAMPLING_TOOL] if PREFERRED_SUBSAMPLING_TOOL.is_file() else attached_files_named("flow_subsampling")
if not tool_candidates:
    raise FileNotFoundError("Attach the extracted official flow_subsampling Linux executable.")
TOOL = WORK_ROOT / "flow_subsampling"
shutil.copy2(tool_candidates[0], TOOL)
TOOL.chmod(0o755)
print("Subsampling tool:", TOOL)

Attached checkpoint: /kaggle/input/datasets/syedmohaiminulhoque/ckpoint/raft-sintel-fb44381e.ckpt
Subsampling tool: /kaggle/working/flow_subsampling


## 6. Run native-resolution inference

The command below invokes the devkit's optical-flow `test.py`. `spring` selects the clean test loader, `raft` selects the full registered RAFT class, the resolved checkpoint supplies Sintel weights, and `model.iters=32` fixes recurrent depth. `predict_all_directions=true` makes the model estimate both frame orders.

No `--max_forward_side` or `--scale_factor` argument is passed, so images are not rescaled. `num_gpus=2` makes the devkit copy the model to both T4s and split samples between them. Each camera side is processed independently. Predictions are written to `/kaggle/temp` as `.flo5`; visualization is disabled. The hard timeout preserves 45 minutes for verification and packaging.

In [6]:
OPTICAL_FLOW_DIR = DEVKIT_DIR / "roco_spring_devkit" / "optical_flow"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = devkit_path + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

cmd = [
    sys.executable, "test.py",
    "--data.test_dataset", "spring",
    "--data.spring_root_dir", str(SPRING_ROOT),
    "--model", MODEL,
    "--ckpt_path", CKPT_ARG,
    "--model.corr_mode", CORR_MODE,
    "--model.iters", str(ITERATIONS),
    "--model.predict_all_directions", "true",
    "--num_gpus", str(NUM_GPUS),
    "--output_path", str(OUTPUT_BASE),
]
assert MAX_FORWARD_SIDE is None
assert "--max_forward_side" not in cmd and "--scale_factor" not in cmd
print("Command:", " ".join(cmd))

elapsed = time.monotonic() - SESSION_START
timeout_seconds = MAX_SESSION_HOURS * 3600 - PACKAGING_RESERVE_MINUTES * 60 - elapsed
if timeout_seconds <= 0:
    raise TimeoutError("No safe inference time remains. Restart the Kaggle session.")
subprocess.run(cmd, cwd=OPTICAL_FLOW_DIR, env=env, check=True, timeout=timeout_seconds)

Command: /usr/bin/python3 test.py --data.test_dataset spring --data.spring_root_dir /kaggle/temp/exp01_spring_merged --model raft --ckpt_path /kaggle/input/datasets/syedmohaiminulhoque/ckpoint/raft-sintel-fb44381e.ckpt --model.corr_mode triton --model.iters 32 --model.predict_all_directions true --num_gpus 2 --output_path /kaggle/temp/exp01_predictions


2026-09-10 04:14:33.031 | WARNING  | roco_spring_devkit.scene_flow.models.raft_3d.raft_3d:<module>:12 - lietorch not found. Please install lietorch from https://github.com/princeton-vl/lietorch if you want to use RAFT3D
2026-09-10 04:14:33.036 | WARNING  | roco_spring_devkit.scene_flow.models.raft_3d.blocks.grid:<module>:13 - scikit-sparse not found. Please install scikit-sparse if you want to use RAFT3D.
/kaggle/working/roco-spring-devkit/roco_spring_devkit/common/data/optical_flow_transforms.py:142: SyntaxWarning: invalid escape sequence '\|'
  In other words, a pixel p is considered occluded when \|Ff(p) + Fb(p + F(f))\|_2 > threshold,
/kaggle/working/roco-spring-devkit/roco_spring_devkit/common/data/optical_flow_transforms.py:160: SyntaxWarning: invalid escape sequence '\|'
  A pixel is considered occluded if \|Ff(p) + Fb(p + F(f))\|_2 > threshold.
/usr/local/lib/python3.12/dist-packages/lightning/fabric/utilities/seed.py:44: No seed found, seed set to 0
Seed set to 0
2026-09-10 04

CompletedProcess(args=['/usr/bin/python3', 'test.py', '--data.test_dataset', 'spring', '--data.spring_root_dir', '/kaggle/temp/exp01_spring_merged', '--model', 'raft', '--ckpt_path', '/kaggle/input/datasets/syedmohaiminulhoque/ckpoint/raft-sintel-fb44381e.ckpt', '--model.corr_mode', 'triton', '--model.iters', '32', '--model.predict_all_directions', 'true', '--num_gpus', '2', '--output_path', '/kaggle/temp/exp01_predictions'], returncode=0)

## 7. Validate predictions before packaging

A completed process is not automatically a valid submission. This stage confirms that inference produced exactly one output tree and the expected number of files. Every filename must follow Spring's `flow_{FW|BW}_{left|right}_####.flo5` convention, all four direction/view combinations must exist, and every HDF5 `flow` dataset must have native shape `1080×1920×2`. Representative files are fully read to reject NaN or infinite vectors.

In [7]:
# Locate and validate the prediction tree before packaging.
import h5py
import numpy as np

prediction_roots = []
for path in OUTPUT_BASE.glob("*/spring"):
    if any(path.rglob("*.flo5")):
        prediction_roots.append(path)
if len(prediction_roots) != 1:
    raise RuntimeError(f"Expected one Spring prediction tree; found {prediction_roots}")
PREDICTION_ROOT = prediction_roots[0]
files = sorted(PREDICTION_ROOT.rglob("*.flo5"))
if len(files) != expected_flow_files:
    raise RuntimeError(f"Expected {expected_flow_files} flow files; found {len(files)}")
name_re = re.compile(r"flow_(FW|BW)_(left|right)_\d{4}\.flo5$")
directions = set()
for path in files:
    match = name_re.fullmatch(path.name)
    if not match:
        raise RuntimeError(f"Invalid prediction filename: {path}")
    directions.add((match.group(1), match.group(2)))
    with h5py.File(path, "r") as handle:
        if "flow" not in handle or handle["flow"].shape != (1080, 1920, 2):
            raise RuntimeError(f"Invalid native-resolution flow shape in {path}: {handle.get('flow').shape if 'flow' in handle else None}")

required = {(d, s) for d in ("FW", "BW") for s in ("left", "right")}
if directions != required:
    raise RuntimeError(f"Expected FW/BW and left/right predictions; found {sorted(directions)}")
for index in sorted({0, len(files) // 2, len(files) - 1}):
    with h5py.File(files[index], "r") as handle:
        if not np.isfinite(handle["flow"][:]).all():
            raise RuntimeError(f"NaN or Inf detected in {files[index]}")
size_gib = sum(p.stat().st_size for p in files) / 1024**3
print(f"Validated {len(files):,} native-resolution .flo5 files ({size_gib:.2f} GiB)")
print("Prediction root:", PREDICTION_ROOT)

Validated 3,960 native-resolution .flo5 files (48.66 GiB)
Prediction root: /kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring


## 8. Build the benchmark artifact

The official `flow_subsampling` program converts the complete prediction directory into the single HDF5 file accepted by the Spring benchmark. The raw `.flo5` tree remains in temporary storage. A JSON manifest records the exact model, checkpoint, iteration count, resolution, devkit revision, file count, artifact size, and SHA-256 checksum so the submitted result can be reproduced and audited.

In [8]:
# Package with the official tool and create a reproducibility manifest.
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([str(TOOL), str(PREDICTION_ROOT)], cwd=ARTIFACT_DIR, check=True)
submission_files = sorted(ARTIFACT_DIR.glob("*.hdf5"), key=lambda p: p.stat().st_mtime)
if not submission_files:
    raise RuntimeError("flow_subsampling did not generate an HDF5 artifact.")
SUBMISSION_FILE = submission_files[-1]

def sha256(path: Path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "team_id": TEAM_ID,
    "team_name": TEAM_NAME,
    "experiment": "01_raft_sintel_32_native",
    "model": MODEL,
    "model_class": "full RAFT",
    "parameter_count": parameter_count,
    "checkpoint": CKPT_ARG,
    "checkpoint_url": expected_checkpoint_url,
    "iterations": ITERATIONS,
    "correlation": CORR_MODE,
    "correlation_levels": 4,
    "correlation_radius": 4,
    "predict_all_directions": True,
    "resolution": "1920x1080_native",
    "devkit_ref": DEVKIT_REF,
    "prediction_files": len(files),
    "submission_file": SUBMISSION_FILE.name,
    "submission_bytes": SUBMISSION_FILE.stat().st_size,
    "submission_sha256": sha256(SUBMISSION_FILE),
}
manifest_path = ARTIFACT_DIR / "experiment_01_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(manifest_path.read_text())
print("UPLOAD THIS FILE:", SUBMISSION_FILE)
print("Also save:", manifest_path)

/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0001.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0002.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0003.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0004.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0005.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0006.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0007.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0008.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381e/spring/0003/flow_FW_left/flow_FW_left_0009.flo5
/kaggle/temp/exp01_predictions/raft_raft-sintel-fb44381

## Submission checklist

- Download the generated `.hdf5` and `experiment_01_manifest.json` from the Kaggle Output pane.
- Upload the `.hdf5` under an Optical Flow method on the [Spring benchmark](https://spring-benchmark.org/).
- Keep the result private until you choose to publish it.
- Record the benchmark method name, submission time and returned clean metrics in your experiment table.
- The workshop-paper abstract must contain the exact team identity: **RoCo-45, Flow State**.